In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm, colors
import scipy as sp
from sympy import *
%matplotlib ipympl 

In [2]:
delta = 8/9

def A(x):
	return np.sqrt(1-delta**2*(1/np.cosh(x))**2)

r_minus_max = -A(0)
r_plus_min = A(0)

r_minus_max, r_plus_min

(-0.45812284729085123, 0.45812284729085123)

In [3]:
X, Y = np.meshgrid(np.linspace(-2,2,100), np.linspace(-2,2,100))
Z = X + 1j*Y

In [102]:
def integrand(eta,endpts,p, branch:str):
	alpha = endpts[0]
	beta = endpts[1]
	if branch=='neg':
		# return eta**p*np.sqrt(eta-alpha)/np.sqrt(-(1-eta**2)*(eta-beta))*(p+1 + (eta*(3*eta**2-2*beta*eta-1)/(2*(1-eta**2)*(eta-beta))))
		return np.sqrt(eta-alpha)*( ((p+1)*eta**p)/np.sqrt(-(1-eta**2)*(eta - beta)) - eta**(p+1)*(3*eta**2-2*beta*eta-1)/(2*((1-eta**2)*(beta-eta))**(3/2) ))
	
	if branch=='pos':
		# return eta**p*np.sqrt(beta-eta)/np.sqrt((1-eta**2)*(eta-alpha))*(p+1 + (eta*(3*eta**2-2*alpha*eta-1)/(2*(1-eta**2)*(eta-alpha))))
		return np.sqrt(beta - eta)*( ((p+1)*eta**p)/np.sqrt((1-eta**2)*(eta - alpha)) -  eta**(p+1)*(-3*eta**2+2*alpha*eta+1)/(2*((1-eta**2)*(eta-alpha))**(3/2)))
	else: 
		return ValueError('Need to specify branch.')


def total_moments(endpts:list,p,x,t, error=False):
	alpha = endpts[0]
	beta = endpts[1]
	
	# minus_moment = r_minus_max**(p+1)*np.sqrt(r_minus_max - alpha)/np.sqrt(-(1-r_minus_max**2)*(r_minus_max-beta)) - sp.integrate.quad(integrand, alpha,r_minus_max, args=(endpts,p,'neg'),points=alpha)[0]
	# plus_moment = -r_plus_min**(p+1)*np.sqrt(beta-r_plus_min)/np.sqrt((1-r_plus_min**2)*(r_plus_min-alpha)) - sp.integrate.quad(integrand, r_plus_min,beta, args=(endpts,p,'pos'), points=beta)[0]
	minus_moment = r_minus_max**(p+1)*np.sqrt(r_minus_max - alpha)/np.sqrt((1-r_minus_max**2)*(beta-r_minus_max)) - sp.integrate.quad(integrand, alpha,r_minus_max, args=(endpts,p,'neg'),points=alpha)[0]
	plus_moment = -r_plus_min**(p+1)*np.sqrt(beta-r_plus_min)/np.sqrt((1-r_plus_min**2)*(r_plus_min-alpha)) + sp.integrate.quad(integrand, r_plus_min,beta, args=(endpts,p,'pos'), points=beta)[0]
	sum_of_moments = minus_moment + plus_moment

	if p==0:
		result = sum_of_moments + x+ (alpha+beta)*t
	
	if p==1:
		result = sum_of_moments + (1/2)*( (alpha+beta)*x + ( (3/2)*alpha**2 + (3/2)*beta**2 + alpha*beta )*t)

	# if error==True:
	# 	print(f"The error from numerically integrating the p={p} moment is {sum_of_moments[1]}.")
	
	return result
	# return np.array([np.real(result),np.imag(result).imag])



def total_moments_system(endpts,x,t, error=False):
	matrix = np.array([total_moments(endpts,0,x,t), total_moments(endpts,1,x,t)])
	return matrix



In [60]:
x_window = 0
t_window = 5
X = np.linspace(0, x_window, 20)
T = np.linspace(0, t_window, 20)

In [101]:
# testing for t=0 at
x0 = 0
t0= 2
alpha = -A(x0)
beta = A(x0)
endpts = [alpha,beta]
# endpts,total_moments_system([alpha,beta],x0,t0)
# sp.optimize.newton(total_moments_system, [-A(x0),A(x0)], args=(x0,t0))

endpts, sp.optimize.fsolve(total_moments_system, [-A(x0),A(x0)], args=(x0,t0))


/var/folders/x8/s1k45xzs5vn1ms0n4nkfqykw0000gn/T/ipykernel_1312/1618292721.py:21: RuntimeWarning: invalid value encountered in sqrt
  minus_moment = r_minus_max**(p+1)*np.sqrt(r_minus_max - alpha)/np.sqrt((1-r_minus_max**2)*(beta-r_minus_max)) - sp.integrate.quad(integrand, alpha,r_minus_max, args=(endpts,p,'neg'),points=alpha)[0]
/var/folders/x8/s1k45xzs5vn1ms0n4nkfqykw0000gn/T/ipykernel_1312/1618292721.py:6: RuntimeWarning: invalid value encountered in sqrt
  return np.sqrt(eta-alpha)*( ((p+1)*eta**p)/np.sqrt(-(1-eta**2)*(eta - beta)) - eta**(p+1)*(3*eta**2-2*beta*eta-1)/(2*((1-eta**2)*(beta-eta))**(3/2) ))
/var/folders/x8/s1k45xzs5vn1ms0n4nkfqykw0000gn/T/ipykernel_1312/1618292721.py:21: IntegrationWarning: The occurrence of roundoff error is detected, which prevents 
  the requested tolerance from being achieved.  The error may be 
  underestimated.
  minus_moment = r_minus_max**(p+1)*np.sqrt(r_minus_max - alpha)/np.sqrt((1-r_minus_max**2)*(beta-r_minus_max)) - sp.integrate.quad(int

([-0.45812284729085123, 0.45812284729085123],
 array([-0.45812285,  0.45812285]))

In [74]:
total_moments_system([-A(x0),A(x0)],x0,t0)

array([9.37162654, 9.48624789])

In [90]:
-r_plus_min**(p+1)*np.sqrt(beta-r_plus_min)/np.sqrt((1-r_plus_min**2)*(r_plus_min-alpha))

-0.0